# 1. Configuracion del entorno

Este bloque prepara el entorno de ejecucion, fija semillas y centraliza la configuracion del experimento. La variable `SM_FRAMEWORK` se define antes de importar `segmentation_models` para evitar inconsistencias de backend.


In [ ]:
# Instalacion de dependencias principales para Google Colab.
print("[INFO] Instalando dependencias principales...")
!pip install -q segmentation-models albumentations==1.3.1 openpyxl seaborn > /dev/null
print("[OK] Dependencias instaladas.")

import os

# Configuracion obligatoria antes de importar segmentation_models.
os.environ["SM_FRAMEWORK"] = "tf.keras"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc
import glob
import random
import shutil
import zipfile
from dataclasses import dataclass
from datetime import datetime

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import segmentation_models as sm
import tensorflow as tf
from google.colab import drive
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import KFold
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard

@dataclass(frozen=True)
class Config:
    SEED: int = 42
    IMG_SIZE: int = 512
    ROI_RADIUS: int = 200
    BATCH_SIZE: int = 8
    EPOCHS: int = 50
    BACKBONE: str = "inceptionresnetv2"
    N_SPLITS: int = 5
    CLASSES: int = 3
    LR_START: float = 1e-4
    RAR_NAME: str = "Refuge.rar"
    DRIVE_BASE: str = "/content/drive/MyDrive/TFG_Glaucoma"
    BASE_PATH: str = "/content/dataset_local/Refuge"
    EXTRACT_PATH: str = "/content/dataset_local"
    SAVE_PATH: str = "/content/drive/MyDrive/TFG_Glaucoma/Models_vPro_Fixed"
    CLINICAL_THRESHOLD: float = 0.52

    @property
    def drive_rar_path(self):
        return os.path.join(self.DRIVE_BASE, self.RAR_NAME)

    @property
    def local_rar_path(self):
        return f"/content/{self.RAR_NAME}"

CFG = Config()


def log_info(message):
    print(f"[INFO] {message}")


def log_ok(message):
    print(f"[OK] {message}")


def log_warning(message):
    print(f"[ADVERTENCIA] {message}")


def log_error(message):
    print(f"[ERROR] {message}")


def log_critical(message):
    print(f"[CRITICO] {message}")


def set_environment(seed=CFG.SEED):
    """Fija semillas y configura TensorFlow para mejorar la reproducibilidad."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        policy = mixed_precision.Policy("mixed_float16")
        mixed_precision.set_global_policy(policy)
        log_ok(f"Mixed precision activado con compute_dtype={policy.compute_dtype}.")
    except Exception as exc:
        log_warning(f"No se pudo activar mixed precision. Se usara float32. Detalle: {exc}")

    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            log_warning(f"No se pudo configurar memory growth en GPU. Detalle: {exc}")

    os.makedirs(CFG.SAVE_PATH, exist_ok=True)
    os.makedirs(os.path.join(CFG.SAVE_PATH, "logs"), exist_ok=True)
    log_ok(f"Semilla global fijada: {seed}.")

set_environment()
preprocess_input = sm.get_preprocessing(CFG.BACKBONE)


# 2. Carga y preparacion del dataset

Se mantiene la estructura original en Google Drive y en REFUGE. Antes de indexar imagenes y mascaras se validan las carpetas criticas para detectar problemas de montaje o descompresion de forma temprana.


In [ ]:
def setup_data():
    """Monta Drive, copia el RAR localmente, descomprime REFUGE y valida carpetas criticas."""
    if not os.path.exists("/content/drive"):
        log_info("Montando Google Drive.")
        drive.mount("/content/drive")

    if not os.path.exists(CFG.drive_rar_path):
        raise FileNotFoundError(f"No se encuentra {CFG.RAR_NAME} en {CFG.DRIVE_BASE}.")

    if not os.path.exists(CFG.local_rar_path):
        log_info(f"Copiando {CFG.RAR_NAME} al almacenamiento local de Colab.")
        shutil.copy(CFG.drive_rar_path, CFG.local_rar_path)
    else:
        log_info(f"El archivo local {CFG.local_rar_path} ya existe.")

    if not os.path.exists(CFG.BASE_PATH):
        log_info("Instalando unrar y descomprimiendo REFUGE.")
        os.makedirs(CFG.EXTRACT_PATH, exist_ok=True)
        os.system("apt-get install -y unrar > /dev/null")
        exit_code = os.system(f'unrar x -o+ "{CFG.local_rar_path}" "{CFG.EXTRACT_PATH}" > /dev/null')
        if exit_code != 0:
            raise RuntimeError("La descompresion del dataset REFUGE no finalizo correctamente.")
    else:
        log_info("El dataset REFUGE ya esta descomprimido.")

    masks_zip_path = os.path.join(CFG.BASE_PATH, "Annotation-Training400", "Disc_Cup_Masks.zip")
    masks_target_dir = os.path.join(CFG.BASE_PATH, "Annotation-Training400", "Disc_Cup_Masks")
    if os.path.exists(masks_zip_path) and not os.path.exists(masks_target_dir):
        log_info("Descomprimiendo mascaras anidadas de entrenamiento.")
        with zipfile.ZipFile(masks_zip_path, "r") as zip_ref:
            zip_ref.extractall(os.path.dirname(masks_zip_path))

    validate_dataset_structure()
    log_ok("Carga y verificacion inicial del dataset completadas.")


def validate_dataset_structure():
    """Comprueba que las rutas esperadas de REFUGE existen antes de entrenar o validar."""
    required_paths = [
        os.path.join(CFG.BASE_PATH, "REFUGE-Training400", "Training400"),
        os.path.join(CFG.BASE_PATH, "Annotation-Training400", "Disc_Cup_Masks"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Validation400", "REFUGE-Validation400"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Validation400-GT", "REFUGE-Validation400-GT", "Disc_Cup_Masks"),
    ]

    missing = [path for path in required_paths if not os.path.exists(path)]
    if missing:
        for path in missing:
            log_error(f"Ruta critica no encontrada: {path}")
        raise FileNotFoundError("La estructura del dataset REFUGE no esta completa.")

    for path in required_paths:
        log_ok(f"Ruta verificada: {path}")

# Ejecutar esta celda antes de indexar datos.
setup_data()


# 3. Protocolo experimental y preprocesamiento

El protocolo separa estrictamente entrenamiento interno y validacion externa. `REFUGE-Training400` se utiliza para entrenamiento y validacion interna mediante K-Fold. `REFUGE-Validation400` queda reservado para la validacion externa final.


In [ ]:
def get_all_pairs_robust(dirs_list):
    """Indexa pares imagen-mascara buscando coincidencias exactas o parciales por nombre base."""
    all_pairs = []
    log_info("Indexando pares imagen-mascara.")

    for img_dir, mask_dir in dirs_list:
        if not os.path.exists(img_dir):
            log_warning(f"Carpeta de imagenes no encontrada: {img_dir}")
            continue
        if not os.path.exists(mask_dir):
            log_warning(f"Carpeta de mascaras no encontrada: {mask_dir}")
            continue

        image_paths = sorted(glob.glob(os.path.join(img_dir, "**", "*.jpg"), recursive=True))
        mask_files = sorted(
            glob.glob(os.path.join(mask_dir, "**", "*.bmp"), recursive=True)
            + glob.glob(os.path.join(mask_dir, "**", "*.png"), recursive=True)
        )
        mask_map = {os.path.splitext(os.path.basename(path))[0]: path for path in mask_files}

        for image_path in image_paths:
            image_key = os.path.splitext(os.path.basename(image_path))[0]
            mask_path = mask_map.get(image_key)
            if mask_path is None:
                mask_path = next((value for key, value in mask_map.items() if image_key in key or key in image_key), None)

            if mask_path is None:
                log_warning(f"No se encontro mascara para la imagen: {image_key}")
            else:
                all_pairs.append((image_path, mask_path))

    return np.array(all_pairs, dtype=object)

TRAIN_DIRS = [
    (
        os.path.join(CFG.BASE_PATH, "REFUGE-Training400", "Training400"),
        os.path.join(CFG.BASE_PATH, "Annotation-Training400", "Disc_Cup_Masks"),
    )
]

EXTERNAL_VAL_DIRS = [
    (
        os.path.join(CFG.BASE_PATH, "REFUGE-Validation400", "REFUGE-Validation400"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Validation400-GT", "REFUGE-Validation400-GT", "Disc_Cup_Masks"),
    )
]

train_data = get_all_pairs_robust(TRAIN_DIRS)
external_val_data = get_all_pairs_robust(EXTERNAL_VAL_DIRS)

log_ok(f"Training interno indexado: {len(train_data)} pares.")
log_ok(f"Validacion externa reservada: {len(external_val_data)} pares.")

if len(train_data) == 0:
    raise ValueError("No se encontraron pares de entrenamiento en REFUGE-Training400.")

if len(external_val_data) == 0:
    log_warning("No se encontraron pares de validacion externa. No se podra cerrar la evaluacion final.")

if len(train_data) != 400:
    log_warning(f"Se esperaban aproximadamente 400 pares de entrenamiento; se han indexado {len(train_data)}.")

if len(external_val_data) != 400:
    log_warning(f"Se esperaban aproximadamente 400 pares de validacion externa; se han indexado {len(external_val_data)}.")


# 4. Construccion del pipeline de entrenamiento

Las funciones de preprocesamiento son compartidas por entrenamiento, validacion, inferencia y visualizacion. Esto evita diferencias artificiales entre lo que el modelo aprende y lo que recibe durante la evaluacion.


In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=180, p=0.8),
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=6, p=0.5),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),
        A.OpticalDistortion(distort_limit=0.2, p=0.5),
    ], p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.2),
])


def read_rgb_image(image_path):
    """Lee una imagen RGB y falla de forma explicita si OpenCV no puede abrirla."""
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {image_path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def read_mask(mask_path):
    """Lee una mascara en escala de grises y valida su disponibilidad."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"No se pudo leer la mascara: {mask_path}")
    return mask


def compute_roi_bounds(image, radius=CFG.ROI_RADIUS):
    """Localiza la region de interes alrededor de la papila usando el canal verde."""
    green_channel = image[:, :, 1]
    blurred = cv2.GaussianBlur(green_channel, (41, 41), 0)
    center_x, center_y = cv2.minMaxLoc(blurred)[3]
    height, width = image.shape[:2]
    x1 = max(0, center_x - radius)
    y1 = max(0, center_y - radius)
    x2 = min(width, center_x + radius)
    y2 = min(height, center_y + radius)
    return x1, y1, x2, y2


def encode_refuge_mask(mask):
    """Convierte la mascara REFUGE a clases semanticas: 0=fondo, 1=disco, 2=copa."""
    semantic_mask = np.zeros(mask.shape, dtype=np.uint8)

    # REFUGE puede aparecer con fondo claro u oscuro segun el archivo exportado.
    if mask[0, 0] > 200:
        semantic_mask[(mask > 100) & (mask < 200)] = 1
        semantic_mask[mask < 100] = 2
    else:
        semantic_mask[(mask > 100) & (mask < 200)] = 1
        semantic_mask[mask > 200] = 2

    return semantic_mask


def apply_clahe_rgb(image):
    """Aplica CLAHE sobre la luminancia para estabilizar contraste sin alterar la API de entrada."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def preprocess_image_for_model(image, apply_clahe=True):
    """Preprocesa una imagen ya recortada y redimensionada para el backbone del modelo."""
    if apply_clahe:
        image = apply_clahe_rgb(image)
    return preprocess_input(image.astype(np.float32)).astype(np.float32)


def preprocess_pair(image_path, mask_path, augment=False):
    """Carga, recorta, redimensiona, aumenta y codifica un par imagen-mascara."""
    image = read_rgb_image(image_path)
    mask = read_mask(mask_path)

    x1, y1, x2, y2 = compute_roi_bounds(image)
    image = image[y1:y2, x1:x2]
    mask = mask[y1:y2, x1:x2]

    image = cv2.resize(image, (CFG.IMG_SIZE, CFG.IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (CFG.IMG_SIZE, CFG.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    semantic_mask = encode_refuge_mask(mask)

    if augment:
        augmented = train_transform(image=image, mask=semantic_mask)
        image = augmented["image"]
        semantic_mask = augmented["mask"]

    image = preprocess_image_for_model(image, apply_clahe=True)
    mask_one_hot = tf.keras.utils.to_categorical(semantic_mask, num_classes=CFG.CLASSES)
    return image.astype(np.float32), mask_one_hot.astype(np.float32)


def preprocess_inference_image(image_path):
    """Aplica el mismo recorte, resize, CLAHE y preprocessing usados en entrenamiento."""
    image = read_rgb_image(image_path)
    x1, y1, x2, y2 = compute_roi_bounds(image)
    crop = image[y1:y2, x1:x2]
    resized = cv2.resize(crop, (CFG.IMG_SIZE, CFG.IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    model_input = preprocess_image_for_model(resized, apply_clahe=True)
    return model_input, resized


def load_data_wrapper(image_path, mask_path, augment=False):
    """Adaptador para tf.numpy_function."""
    return preprocess_pair(image_path.decode("utf-8"), mask_path.decode("utf-8"), bool(augment))


def create_dataset(pairs, augment=False):
    """Crea un pipeline tf.data reproducible para segmentacion semantica."""
    image_paths, mask_paths = zip(*pairs)
    dataset = tf.data.Dataset.from_tensor_slices((list(image_paths), list(mask_paths)))
    dataset = dataset.map(
        lambda image_path, mask_path: tf.numpy_function(
            func=load_data_wrapper,
            inp=[image_path, mask_path, augment],
            Tout=[tf.float32, tf.float32],
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    dataset = dataset.map(
        lambda image, mask: (
            tf.ensure_shape(image, [CFG.IMG_SIZE, CFG.IMG_SIZE, 3]),
            tf.ensure_shape(mask, [CFG.IMG_SIZE, CFG.IMG_SIZE, CFG.CLASSES]),
        )
    )
    if augment:
        dataset = dataset.shuffle(buffer_size=min(len(pairs), 400), seed=CFG.SEED)
    return dataset.batch(CFG.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# 5. Definicion del modelo

Se mantiene la arquitectura U-Net con backbone `inceptionresnetv2`. Las metricas historicas `dice_coef_disc` y `dice_coef_cup` se conservan por compatibilidad, pero calculan IoU por clase mediante `segmentation_models.metrics.IOUScore`.


In [ ]:
def cast_f32(tensor):
    return tf.cast(tensor, tf.float32)

IOU_METRIC_OBJ = sm.metrics.IOUScore(threshold=0.5)
DICE_LOSS_OBJ = sm.losses.DiceLoss()
FOCAL_LOSS_OBJ = sm.losses.CategoricalFocalLoss()


def dice_coef_disc(y_true, y_pred):
    """Metrica historica: IoU de disco optico, conservada por compatibilidad de nombres."""
    return IOU_METRIC_OBJ(cast_f32(y_true[..., 1]), cast_f32(y_pred[..., 1]))


def dice_coef_cup(y_true, y_pred):
    """Metrica historica: IoU de copa optica, conservada por compatibilidad de nombres."""
    return IOU_METRIC_OBJ(cast_f32(y_true[..., 2]), cast_f32(y_pred[..., 2]))


def global_iou(y_true, y_pred):
    """IoU global sobre la mascara multiclase."""
    return IOU_METRIC_OBJ(cast_f32(y_true), cast_f32(y_pred))


def hybrid_loss(y_true, y_pred):
    """Funcion de perdida combinada para segmentacion multiclase."""
    y_true = cast_f32(y_true)
    y_pred = cast_f32(y_pred)
    return DICE_LOSS_OBJ(y_true, y_pred) + FOCAL_LOSS_OBJ(y_true, y_pred)


def build_model():
    """Construye y compila U-Net con backbone inceptionresnetv2."""
    model = sm.Unet(
        CFG.BACKBONE,
        encoder_weights="imagenet",
        classes=CFG.CLASSES,
        activation="softmax",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(CFG.LR_START),
        loss=hybrid_loss,
        metrics=[global_iou, dice_coef_disc, dice_coef_cup],
    )
    return model

log_ok("Definicion de modelo y metricas preparada.")


# 6. Entrenamiento K-Fold

El K-Fold se aplica exclusivamente sobre `train_data`, que procede de `REFUGE-Training400`. Esta celda es computacionalmente costosa y puede omitirse si ya existen los cinco modelos finales en `Models_vPro_Fixed`.


In [ ]:
def run_kfold_training():
    """Entrena cinco folds usando solo REFUGE-Training400 y guarda model_fold_*.keras."""
    if len(train_data) < CFG.N_SPLITS:
        raise ValueError("No hay suficientes pares de entrenamiento para ejecutar K-Fold.")

    kfold = KFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_data), start=1):
        log_info("=" * 60)
        log_info(f"Fold {fold}/{CFG.N_SPLITS}. Inicio: {datetime.now().strftime('%H:%M:%S')}.")

        fold_train_pairs = train_data[train_idx]
        fold_val_pairs = train_data[val_idx]
        log_info(f"Imagenes de entrenamiento interno: {len(fold_train_pairs)}.")
        log_info(f"Imagenes de validacion interna: {len(fold_val_pairs)}.")

        train_ds = create_dataset(fold_train_pairs, augment=True)
        val_ds = create_dataset(fold_val_pairs, augment=False)
        model = build_model()

        model_path = os.path.join(CFG.SAVE_PATH, f"model_fold_{fold}.keras")
        log_path = os.path.join(CFG.SAVE_PATH, f"log_fold_{fold}.csv")
        tensorboard_path = os.path.join(CFG.SAVE_PATH, "logs", f"fold_{fold}")

        callbacks = [
            ModelCheckpoint(model_path, monitor="val_global_iou", mode="max", save_best_only=True, verbose=1),
            EarlyStopping(monitor="val_global_iou", mode="max", patience=12, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_global_iou", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
            CSVLogger(log_path),
            TensorBoard(log_dir=tensorboard_path),
        ]

        model.fit(train_ds, validation_data=val_ds, epochs=CFG.EPOCHS, callbacks=callbacks, verbose=1)

        del model, train_ds, val_ds
        tf.keras.backend.clear_session()
        gc.collect()
        log_ok(f"Fold {fold} finalizado. Modelo guardado en {model_path}.")

    log_ok("Entrenamiento K-Fold completado exclusivamente sobre REFUGE-Training400.")

# Descomentar para entrenar. No ejecutar si solo se desea validar modelos ya entrenados.
# run_kfold_training()


# 7. Carga y verificacion de modelos entrenados

Este bloque comprueba y carga los modelos finales `model_fold_1.keras` a `model_fold_5.keras` desde `Models_vPro_Fixed`. No reconstruye datasets mezclados ni modifica el protocolo experimental.


In [ ]:
def get_expected_model_paths():
    return [os.path.join(CFG.SAVE_PATH, f"model_fold_{fold}.keras") for fold in range(1, CFG.N_SPLITS + 1)]


def verify_model_files():
    """Verifica la presencia de los cinco modelos finales esperados."""
    expected_paths = get_expected_model_paths()
    missing_paths = [path for path in expected_paths if not os.path.exists(path)]

    for path in expected_paths:
        if os.path.exists(path):
            log_ok(f"Modelo disponible: {path}")
        else:
            log_error(f"Modelo no encontrado: {path}")

    if missing_paths:
        raise FileNotFoundError("Faltan modelos entrenados en Models_vPro_Fixed.")

    return expected_paths


def enable_keras_unsafe_deserialization_if_needed():
    """Permite cargar modelos guardados con capas internas de segmentation_models en Keras 3."""
    try:
        import keras
        keras.config.enable_unsafe_deserialization()
        log_info("Deserializacion compatible con Keras 3 habilitada.")
    except Exception as exc:
        log_warning(f"No se pudo modificar la politica de deserializacion. Detalle: {exc}")


def load_ensemble_models():
    """Carga en memoria los modelos del ensemble final."""
    model_paths = verify_model_files()
    enable_keras_unsafe_deserialization_if_needed()

    loaded_models = []
    for path in model_paths:
        log_info(f"Cargando modelo: {os.path.basename(path)}")
        try:
            model = tf.keras.models.load_model(path, compile=False, safe_mode=False)
        except TypeError:
            model = tf.keras.models.load_model(path, compile=False)
        loaded_models.append(model)

    log_ok(f"Ensemble cargado con {len(loaded_models)} modelos.")
    return loaded_models

# Ejecutar para validar o inferir.
# models = load_ensemble_models()


# 8. Inferencia y segmentacion

La inferencia usa el mismo preprocesamiento que el entrenamiento, aplica TTA horizontal y postprocesa la mascara para conservar estructuras anatomicas compactas.


In [ ]:
def predict_tta_ensemble(models, input_batch):
    """Calcula la prediccion media del ensemble con TTA horizontal."""
    if not models:
        raise ValueError("No hay modelos cargados para inferencia.")

    predictions = []
    for model in models:
        predictions.append(model.predict(input_batch, verbose=0))

    flipped_batch = np.flip(input_batch, axis=2)
    for model in models:
        flipped_prediction = model.predict(flipped_batch, verbose=0)
        predictions.append(np.flip(flipped_prediction, axis=2))

    return np.mean(predictions, axis=0)


def keep_largest_component(binary_mask):
    """Conserva el mayor componente conectado de una mascara binaria."""
    binary_mask = binary_mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    if num_labels <= 1:
        return binary_mask

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label).astype(np.uint8)


def postprocess_anatomical_mask(mask):
    """Limpia la segmentacion respetando restricciones anatomicas basicas disco-copa."""
    if mask.ndim == 3:
        mask = mask[0]

    disc = ((mask == 1) | (mask == 2)).astype(np.uint8)
    cup = (mask == 2).astype(np.uint8)

    if np.sum(disc) == 0:
        log_warning("Segmentacion sin disco optico. Se devuelve mascara vacia.")
        return np.zeros_like(mask, dtype=np.uint8)

    disc = keep_largest_component(disc)
    cup = keep_largest_component(cup) if np.sum(cup) > 0 else cup
    cup = (cup & disc).astype(np.uint8)

    cleaned_mask = np.zeros_like(mask, dtype=np.uint8)
    cleaned_mask[disc == 1] = 1
    cleaned_mask[cup == 1] = 2
    return cleaned_mask


def segment_image(models, image_path):
    """Ejecuta inferencia completa sobre una imagen y devuelve mascara, probabilidades e imagen preprocesada visible."""
    model_input, visible_image = preprocess_inference_image(image_path)
    input_batch = np.expand_dims(model_input, axis=0)
    probabilities = predict_tta_ensemble(models, input_batch)
    raw_mask = np.argmax(probabilities, axis=-1)[0]
    mask = postprocess_anatomical_mask(raw_mask)
    return mask, probabilities[0], visible_image


# 9. Extraccion de biomarcadores clinicos

Los biomarcadores se calculan a partir de la mascara postprocesada. El indicador ISNT se documenta como indicador geometrico simplificado inspirado en la regla ISNT, no como una evaluacion clinica completa.


In [ ]:
def vertical_diameter(binary_mask):
    rows = np.where(np.any(binary_mask, axis=1))[0]
    if rows.size == 0:
        return 0
    return int(rows[-1] - rows[0] + 1)


def safe_ratio(numerator, denominator):
    if denominator <= 0:
        return np.nan
    return float(numerator) / float(denominator)


def compute_simplified_isnt_indicator(disc_mask, cup_mask):
    """Calcula un indicador geometrico simplificado inspirado en la regla ISNT."""
    if np.sum(disc_mask) == 0 or np.sum(cup_mask) == 0:
        return np.nan

    moments = cv2.moments(disc_mask.astype(np.uint8))
    if moments["m00"] == 0:
        return np.nan

    center_x = int(moments["m10"] / moments["m00"])
    center_y = int(moments["m01"] / moments["m00"])

    rim = (disc_mask.astype(bool) & ~cup_mask.astype(bool))
    inferior = np.sum(rim[center_y:, :])
    superior = np.sum(rim[:center_y, :])
    nasal = np.sum(rim[:, :center_x])
    temporal = np.sum(rim[:, center_x:])

    total_rim = inferior + superior + nasal + temporal
    if total_rim == 0:
        return np.nan

    # Puntuacion simple: aumenta cuando inferior y superior son relativamente conservados.
    vertical_rim = inferior + superior
    horizontal_rim = nasal + temporal
    return float(np.clip(vertical_rim / (vertical_rim + horizontal_rim), 0.0, 1.0))


def compute_biomarkers(mask):
    """Extrae biomarcadores y controla segmentaciones invalidas o geometricamente imposibles."""
    if mask.ndim == 3:
        mask = mask[0]

    disc_mask = ((mask == 1) | (mask == 2)).astype(np.uint8)
    cup_mask = (mask == 2).astype(np.uint8)

    disc_diameter = vertical_diameter(disc_mask)
    cup_diameter = vertical_diameter(cup_mask)
    disc_area = int(np.sum(disc_mask))
    cup_area = int(np.sum(cup_mask))

    vcdr = safe_ratio(cup_diameter, disc_diameter)
    rcdr = safe_ratio(cup_area, disc_area)
    isnt_indicator = compute_simplified_isnt_indicator(disc_mask, cup_mask)

    warnings = []
    if disc_area == 0:
        warnings.append("Disco optico no detectado")
    if cup_area == 0:
        warnings.append("Copa optica no detectada")
    if cup_area > disc_area:
        warnings.append("Area de copa superior al area de disco")
    if np.isnan(vcdr) or vcdr < 0 or vcdr > 1:
        warnings.append("vCDR fuera de rango valido")
    if np.isnan(rcdr) or rcdr < 0 or rcdr > 1:
        warnings.append("rCDR fuera de rango valido")

    return {
        "disc_vertical_diameter": disc_diameter,
        "cup_vertical_diameter": cup_diameter,
        "disc_area": disc_area,
        "cup_area": cup_area,
        "vcdr": vcdr,
        "rcdr": rcdr,
        "isnt_indicator": isnt_indicator,
        "warnings": warnings,
    }


def normalize_metric(value, lower, width):
    if value is None or np.isnan(value):
        return 0.0
    return float(np.clip((value - lower) / width, 0.0, 1.0))


def compute_risk_scores(biomarkers):
    """Calcula scores comparables para validacion y ablacion."""
    vcdr_score = normalize_metric(biomarkers["vcdr"], 0.35, 0.40)
    rcdr_score = normalize_metric(biomarkers["rcdr"], 0.25, 0.40)
    isnt_score = 1.0 - biomarkers["isnt_indicator"] if not np.isnan(biomarkers["isnt_indicator"]) else 0.0

    return {
        "vcdr_only": vcdr_score,
        "vcdr_rcdr": float(np.clip(0.70 * vcdr_score + 0.30 * rcdr_score, 0.0, 1.0)),
        "final": float(np.clip(0.60 * vcdr_score + 0.25 * rcdr_score + 0.15 * isnt_score, 0.0, 1.0)),
    }


# 10. Validacion externa

La validacion externa se ejecuta exclusivamente sobre `external_val_data`, que procede de `REFUGE-Validation400`. El umbral clinico se mantiene como criterio heuristico y debe justificarse o recalibrarse antes de presentar metricas definitivas.


In [ ]:
def load_ground_truth_labels():
    """Carga etiquetas reales de REFUGE-Validation400 desde el fichero clinico disponible."""
    candidates = [
        os.path.join(CFG.BASE_PATH, "REFUGE-Validation400-GT", "REFUGE-Validation400-GT", "Fovea_locations.xlsx"),
        os.path.join(CFG.BASE_PATH, "REFUGE-Validation400-GT", "REFUGE-Validation400-GT", "Fovea_locations.csv"),
    ]
    label_path = next((path for path in candidates if os.path.exists(path)), None)
    if label_path is None:
        raise FileNotFoundError("No se encontro fichero de etiquetas clinicas de validacion externa.")

    dataframe = pd.read_excel(label_path) if label_path.endswith(".xlsx") else pd.read_csv(label_path)
    name_candidates = [column for column in dataframe.columns if "Img" in column or "Name" in column or "image" in column.lower()]
    if not name_candidates:
        raise ValueError("No se encontro columna de nombre de imagen en el fichero clinico.")
    if "Glaucoma Label" not in dataframe.columns:
        raise ValueError("No se encontro la columna 'Glaucoma Label'.")

    name_column = name_candidates[0]
    labels = {
        os.path.splitext(str(row[name_column]))[0]: int(row["Glaucoma Label"])
        for _, row in dataframe.iterrows()
    }
    log_ok(f"Etiquetas clinicas cargadas: {len(labels)} registros desde {label_path}.")
    return labels


def evaluate_external_validation(models, threshold=CFG.CLINICAL_THRESHOLD):
    """Evalua el ensemble sobre REFUGE-Validation400 y devuelve resultados por caso y metricas."""
    if len(external_val_data) == 0:
        raise ValueError("external_val_data esta vacio. No se puede ejecutar validacion externa.")

    gt_labels = load_ground_truth_labels()
    records = []

    log_info(f"Iniciando validacion externa sobre {len(external_val_data)} pares.")
    for index, (image_path, _) in enumerate(external_val_data, start=1):
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        if image_id not in gt_labels:
            log_warning(f"Imagen sin etiqueta clinica; se omite: {image_id}")
            continue

        mask, _, _ = segment_image(models, image_path)
        biomarkers = compute_biomarkers(mask)
        scores = compute_risk_scores(biomarkers)

        records.append({
            "image_id": image_id,
            "y_true": gt_labels[image_id],
            "vcdr": biomarkers["vcdr"],
            "rcdr": biomarkers["rcdr"],
            "isnt_indicator": biomarkers["isnt_indicator"],
            "score_vcdr_only": scores["vcdr_only"],
            "score_vcdr_rcdr": scores["vcdr_rcdr"],
            "score_final": scores["final"],
            "warnings": "; ".join(biomarkers["warnings"]),
        })

        if index % 25 == 0:
            log_info(f"Casos procesados: {index}/{len(external_val_data)}.")

    results = pd.DataFrame(records)
    if results.empty:
        raise ValueError("No se genero ningun resultado de validacion externa.")

    metrics = compute_classification_metrics(results["y_true"].values, results["score_final"].values, threshold)
    print_external_report(metrics, threshold)
    plot_confusion_matrix(metrics["confusion_matrix"], threshold)
    return results, metrics


def compute_classification_metrics(y_true, y_score, threshold):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    y_true = np.asarray(y_true).astype(int)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()

    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan

    return {
        "auc_roc": auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1,
        "confusion_matrix": matrix,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def print_external_report(metrics, threshold):
    log_info("=" * 60)
    log_info("REPORTE DE VALIDACION EXTERNA")
    log_info(f"Umbral aplicado: {threshold:.4f}")
    log_info(f"AUC-ROC: {metrics['auc_roc']:.4f}")
    log_info(f"Sensibilidad: {metrics['sensitivity']:.4f}")
    log_info(f"Especificidad: {metrics['specificity']:.4f}")
    log_info(f"Precision: {metrics['precision']:.4f}")
    log_info(f"F1-score: {metrics['f1_score']:.4f}")
    log_info(f"Matriz de confusion: TN={metrics['tn']}, FP={metrics['fp']}, FN={metrics['fn']}, TP={metrics['tp']}")


def plot_confusion_matrix(matrix, threshold):
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Pred. sano", "Pred. glaucoma"],
        yticklabels=["Real sano", "Real glaucoma"],
    )
    plt.title(f"Matriz de confusion externa. Umbral={threshold:.2f}")
    plt.xlabel("Prediccion")
    plt.ylabel("Etiqueta real")
    plt.tight_layout()
    plt.show()

# Ejecucion recomendada tras cargar modelos:
# models = load_ensemble_models()
# external_results, external_metrics = evaluate_external_validation(models)


# 11. Estudio de ablacion

La ablacion compara configuraciones de biomarcadores sobre el mismo conjunto externo: solo `vCDR`, `vCDR + rCDR` y `vCDR + rCDR + indicador ISNT simplificado`.


In [ ]:
def run_ablation_study(external_results):
    """Compara AUC-ROC de distintas combinaciones de biomarcadores en validacion externa."""
    required_columns = ["y_true", "score_vcdr_only", "score_vcdr_rcdr", "score_final"]
    missing_columns = [column for column in required_columns if column not in external_results.columns]
    if missing_columns:
        raise ValueError(f"Faltan columnas para la ablacion: {missing_columns}")

    variants = {
        "Solo vCDR": external_results["score_vcdr_only"].values,
        "vCDR + rCDR": external_results["score_vcdr_rcdr"].values,
        "vCDR + rCDR + ISNT simplificado": external_results["score_final"].values,
    }

    y_true = external_results["y_true"].values
    rows = []
    plt.figure(figsize=(8, 6))

    for label, scores in variants.items():
        auc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else np.nan
        rows.append({"variante": label, "auc_roc": auc})
        fpr, tpr, _ = roc_curve(y_true, scores)
        plt.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC={auc:.3f})")

    plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar")
    plt.xlabel("Tasa de falsos positivos")
    plt.ylabel("Sensibilidad")
    plt.title("Estudio de ablacion en validacion externa")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    ablation_table = pd.DataFrame(rows).sort_values("auc_roc", ascending=False)
    log_info("Resultado del estudio de ablacion:")
    display(ablation_table)
    return ablation_table

# Ejecutar despues de evaluate_external_validation:
# ablation_results = run_ablation_study(external_results)


# 12. Validacion visual

La validacion visual se situa al final porque sirve para interpretar aciertos y errores despues de cerrar la evaluacion cuantitativa.


In [ ]:
def risk_label(score, threshold=CFG.CLINICAL_THRESHOLD):
    return "Glaucoma probable" if score >= threshold else "Sano probable"


def plot_visual_case(models, image_path, y_true=None, threshold=CFG.CLINICAL_THRESHOLD):
    """Muestra imagen, mascara postprocesada, superposicion y biomarcadores de un caso."""
    mask, _, visible_image = segment_image(models, image_path)
    biomarkers = compute_biomarkers(mask)
    scores = compute_risk_scores(biomarkers)
    final_score = scores["final"]

    overlay = visible_image.copy()
    disc_color = np.array([255, 180, 0], dtype=np.uint8)
    cup_color = np.array([220, 40, 40], dtype=np.uint8)
    alpha = 0.35
    overlay[mask == 1] = ((1 - alpha) * overlay[mask == 1] + alpha * disc_color).astype(np.uint8)
    overlay[mask == 2] = ((1 - alpha) * overlay[mask == 2] + alpha * cup_color).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(visible_image)
    axes[0].set_title("Imagen preprocesada")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="viridis", vmin=0, vmax=2)
    axes[1].set_title("Mascara disco-copa")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Superposicion")
    axes[2].axis("off")

    label_text = "No disponible" if y_true is None else str(y_true)
    fig.suptitle(
        " | ".join([
            f"Score={final_score:.3f}",
            f"Decision={risk_label(final_score, threshold)}",
            f"Etiqueta real={label_text}",
            f"vCDR={biomarkers['vcdr']:.3f}" if not np.isnan(biomarkers['vcdr']) else "vCDR=NaN",
            f"rCDR={biomarkers['rcdr']:.3f}" if not np.isnan(biomarkers['rcdr']) else "rCDR=NaN",
        ]),
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    if biomarkers["warnings"]:
        log_warning("; ".join(biomarkers["warnings"]))


def plot_representative_cases(models, external_results, max_cases=5):
    """Selecciona casos representativos: aciertos, errores y casos cercanos al umbral."""
    if external_results.empty:
        raise ValueError("No hay resultados externos para seleccionar casos visuales.")

    data = external_results.copy()
    data["prediction"] = (data["score_final"] >= CFG.CLINICAL_THRESHOLD).astype(int)
    data["distance_to_threshold"] = np.abs(data["score_final"] - CFG.CLINICAL_THRESHOLD)

    selected_ids = []
    selection_rules = [
        data[(data["y_true"] == 0) & (data["prediction"] == 0)],
        data[(data["y_true"] == 1) & (data["prediction"] == 1)],
        data[(data["y_true"] == 0) & (data["prediction"] == 1)],
        data[(data["y_true"] == 1) & (data["prediction"] == 0)],
        data.sort_values("distance_to_threshold"),
    ]

    for subset in selection_rules:
        if not subset.empty:
            selected_ids.append(subset.iloc[0]["image_id"])

    selected_ids = list(dict.fromkeys(selected_ids))[:max_cases]
    image_map = {os.path.splitext(os.path.basename(path))[0]: path for path, _ in external_val_data}

    for image_id in selected_ids:
        row = data[data["image_id"] == image_id].iloc[0]
        log_info(f"Visualizando caso {image_id}: y_true={row['y_true']}, score={row['score_final']:.3f}.")
        plot_visual_case(models, image_map[image_id], y_true=row["y_true"])

# Ejecutar despues de validar:
# plot_representative_cases(models, external_results)


# 13. Resumen final del experimento

Este notebook queda organizado como pipeline tecnico reproducible: datos, protocolo, preprocesamiento, modelo, entrenamiento, inferencia, biomarcadores, validacion externa, ablacion y validacion visual.

Puntos metodologicos fijados:

- `REFUGE-Training400` se usa para entrenamiento y validacion interna K-Fold.
- `REFUGE-Validation400` se reserva para validacion externa final.
- Los modelos finales se guardan como `model_fold_1.keras` a `model_fold_5.keras` en `Models_vPro_Fixed`.
- El indicador ISNT implementado es un indicador geometrico simplificado inspirado en la regla ISNT.
- El umbral `0.52` se mantiene como criterio heuristico hasta recalibracion o justificacion formal.
- Las metricas finales deben recalcularse tras aplicar este protocolo limpio.

Streamlit queda fuera del flujo principal hasta cerrar entrenamiento, validacion externa, score final y umbral clinico.
